# Getting Started with Agentick

This notebook provides an interactive introduction to the Agentick benchmark.

You'll learn how to:
- Create and interact with environments
- Use different observation modes
- Run simple agents
- Visualize results

## Setup

First, import the required libraries:

In [ ]:
import numpy as np
from agentick.agents.oracle import OracleAgent

import agentick

## Creating Your First Environment

Let's create a simple navigation task:

In [ ]:
# Create environment
env = agentick.make("GoToGoal-v0", difficulty="easy", render_mode="ascii")

# Reset to get initial observation
obs, info = env.reset(seed=42)

print("Initial observation:")
print(obs)
print(f"\nAction space: {env.action_space}")
print(f"Observation type: {type(obs)}")

## Taking Actions

Let's take a random action and see what happens:

In [ ]:
# Take a random action
action = env.action_space.sample()
print(f"Taking action: {action}")

obs, reward, terminated, truncated, info = env.step(action)

print("\nNew observation:")
print(obs)
print(f"\nReward: {reward}")
print(f"Terminated: {terminated}")
print(f"Truncated: {truncated}")

## Running a Complete Episode

Let's run a full episode with a random agent:

In [ ]:
# Reset environment
obs, info = env.reset(seed=42)

total_reward = 0
steps = 0

# Run episode
for step in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    if terminated or truncated:
        break

print("Episode finished!")
print(f"Total reward: {total_reward}")
print(f"Steps taken: {steps}")
print(f"Success: {info.get('success', False)}")

## Different Observation Modes

Agentick supports multiple observation formats:

In [ ]:
# ASCII mode (text-based grid)
env_ascii = agentick.make("GoToGoal-v0", render_mode="ascii")
obs, _ = env_ascii.reset(seed=42)
print("ASCII observation:")
print(obs)
print()

# Language mode (natural language description)
env_lang = agentick.make("GoToGoal-v0", render_mode="language")
obs, _ = env_lang.reset(seed=42)
print("Language observation:")
print(obs)
print()

# RGB array (pixel observations)
env_rgb = agentick.make("GoToGoal-v0", render_mode="rgb_array")
obs, _ = env_rgb.reset(seed=42)
print("RGB observation shape:", obs.shape)

# Clean up
env_ascii.close()
env_lang.close()
env_rgb.close()

## Using the Oracle Agent

The oracle agent knows the optimal policy:

In [ ]:
# Create environment and oracle agent
env = agentick.make("GoToGoal-v0", difficulty="easy", render_mode="ascii")
oracle = OracleAgent(env)

# Run oracle for 5 episodes
print("Running oracle agent...\n")
results = []

for episode in range(5):
    obs, info = env.reset(seed=100 + episode)
    total_reward = 0
    steps = 0

    for _ in range(100):
        action = oracle.act(obs, info)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1

        if terminated or truncated:
            break

    success = info.get("success", False)
    results.append({"reward": total_reward, "steps": steps, "success": success})
    print(f"Episode {episode + 1}: reward={total_reward:.2f}, steps={steps}, success={success}")

# Summary statistics
avg_reward = np.mean([r["reward"] for r in results])
success_rate = np.mean([r["success"] for r in results])
avg_steps = np.mean([r["steps"] for r in results])

print("\nOracle Performance:")
print(f"Average reward: {avg_reward:.2f}")
print(f"Success rate: {success_rate:.1%}")
print(f"Average steps: {avg_steps:.1f}")

env.close()

## Exploring Different Tasks

Let's list all available tasks:

In [ ]:
from agentick.tasks.registry import list_tasks

tasks = list_tasks()
print(f"Total tasks: {len(tasks)}\n")
print("First 10 tasks:")
for i, task in enumerate(tasks[:10], 1):
    print(f"  {i}. {task}")

## Difficulty Levels

Each task has 4 difficulty levels:

In [ ]:
difficulties = ["easy", "medium", "hard", "expert"]

print("Running oracle on different difficulties:\n")

for difficulty in difficulties:
    env = agentick.make("GoToGoal-v0", difficulty=difficulty, render_mode="ascii")
    oracle = OracleAgent(env)

    # Run 3 episodes
    rewards = []
    for ep in range(3):
        obs, info = env.reset(seed=200 + ep)
        total_reward = 0

        for _ in range(100):
            action = oracle.act(obs, info)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        rewards.append(total_reward)

    avg_reward = np.mean(rewards)
    print(f"{difficulty:8s}: avg reward = {avg_reward:.2f}")

    env.close()

## Next Steps

Now that you understand the basics, check out:

1. **02_analyze_experiment.ipynb** - Explore experiment data and plots
2. **03_compare_agents.ipynb** - Multi-agent comparison
3. **examples/rl/** - Train RL agents (PPO, DQN)
4. **examples/llm/** - Use LLM/VLM agents
5. **docs/** - Full documentation

## Clean Up

Always close environments when done:

In [ ]:
env.close()
print("Environment closed!")